# Merging COCO dataset into FiftyOne
Merges updated COCO-format annotations into an existing FiftyOne dataset.
Tracks full statistics: new images, new polylines, label changes, skipped samples.

In [ ]:
import json
import os
import logging
from collections import defaultdict
from datetime import datetime

import fiftyone as fo
import numpy as np
from PIL import Image
from tqdm import tqdm

from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(f"merge_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
    ]
)
log = logging.getLogger(__name__)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
COCO_DS_PATH   = "/home/andrew/Andrew/Fishial2402/dataset/clean_30_03.json"
IMAGES_DIR     = "/home/andrew/Andrew/Fishial2402/dataset/segmentation_dataset_v0.10_with_meta/data"
FO_DS_NAME     = "segmentation_dataset_v0.10_with_meta"
NEW_FO_DS_NAME = "segmentation_dataset_v0.11_with_meta"
FIELD_NAME     = "General body shape"   # FiftyOne field that holds polylines
EXPORT_TAG     = "export_30_03_2026"    # tag applied to all new/modified samples
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# fo_ds_old = fo.load_dataset('segmentation_dataset_v0.11_with_meta')
# fo_ds_old.delete()

In [ ]:
log.info("Loading COCO JSON …")
with open(COCO_DS_PATH) as f:
    COCO_DS_DICT = json.load(f)

log.info("Loading FiftyOne dataset …")
fo_ds_old = fo.load_dataset(FO_DS_NAME)
try:
    fo_ds = fo_ds_old.clone(NEW_FO_DS_NAME)
except Exception as e:
    fo.load_dataset(NEW_FO_DS_NAME).delete()
    fo_ds = fo_ds_old.clone(NEW_FO_DS_NAME)
    print("deteled and created new dataset")
    # pass
log.info(
    "COCO: %d images | %d annotations | %d categories",
    len(COCO_DS_DICT['images']),
    len(COCO_DS_DICT['annotations']),
    len(COCO_DS_DICT['categories']),
)
log.info("FiftyOne dataset: %d samples", len(fo_ds))

In [ ]:
with open('/home/andrew/Andrew/Fishial2402/fish-identification/labels_866.txt', 'r', encoding='utf-8') as f:
    keys_list = f.read().splitlines()

In [ ]:
COCO_DS_DICT['categories']
txt_labels = list()

class_mapping_before = dict()
for category_inst in COCO_DS_DICT['categories']:
    if 'General body shape' != category_inst['name']:
        continue
    
    try:
        class_mapping_before[category_inst['supercategory']] = category_inst['fishial_extra']['species_id']
    except:
        print("error: ", category_inst )

In [ ]:
class_mapping = dict()
for label in keys_list:
    class_mapping[label] = {
        'id': len(class_mapping),
        'fishial_extra': {
            'species_id': class_mapping_before[label],
        }
    }

In [ ]:
with open('class_mapping_866.json', 'w', encoding='utf-8') as f:
    json.dump(class_mapping, f, ensure_ascii=False, indent=4)

In [ ]:
# import os
# import json
# import mimetypes
# from collections import Counter
# from typing import Dict, Any

# # Urgent fix for recognizing webp and other custom types
# # (Kept based on the original script, though not strictly required for os.path.exists)
# mimetypes.add_type("image/webp", ".webp")

# # ---------------------------------------------------------
# # Configuration and Paths
# # ---------------------------------------------------------
# CONFIG = {
#     "images_dir": "/home/andrew/Andrew/Fishial2402/dataset/segmentation_dataset_v0.10_with_meta/data/",
#     "input_coco_path": "/home/andrew/Andrew/Fishial2402/dataset/new_export.json",
#     "output_coco_path": "/home/andrew/Andrew/Fishial2402/dataset/clean_30_03.json",
#     # List of extensions to brute-force on the disk
#     "possible_exts": ['.png', '.jpg', '.jpeg', '.webp', '.jfif', '.gif', '.JPG', '.PNG']
# }


# def fix_image_paths(config: Dict[str, Any]) -> None:
#     """
#     Validates and fixes image paths in a COCO JSON file by matching them 
#     against actual files on disk using a brute-force extension search.
#     """
#     print(f"Loading COCO dataset from: {config['input_coco_path']}")
#     with open(config['input_coco_path'], 'r', encoding='utf-8') as f:
#         coco_dict = json.load(f)

#     fixed_count = 0
#     not_found_count = 0
#     images_dir = config["images_dir"]
#     possible_exts = config["possible_exts"]

#     print("Starting deep file search...")

#     # Main iteration over images
#     for img in coco_dict['images']:
#         old_name = img['file_name']
        
#         # Check if the file exists "as is"
#         if os.path.exists(os.path.join(images_dir, old_name)):
#             continue  # File is already in place, skip
            
#         # If not found, start brute-forcing the extensions
#         # Strip the current extension to get the pure ID/base name
#         base_name = os.path.splitext(old_name)[0]
        
#         # Additionally clean up garbage characters like ';' or '-' if present
#         base_name = base_name.split(';')[0].split('-')[0]
        
#         found = False
#         for ext in possible_exts:
#             potential_path = os.path.join(images_dir, base_name + ext)
            
#             if os.path.exists(potential_path):
#                 # Found it! Update the filename in the JSON to match the disk
#                 img['file_name'] = base_name + ext
#                 fixed_count += 1
#                 found = True
#                 break
                
#         if not found:
#             print(f"CRITICAL: Could not find file for {old_name} (searched as {base_name} + {possible_exts})")
#             not_found_count += 1

#     # Collect final extension statistics
#     extension_counts = Counter()
#     for img in coco_dict['images']:
#         _, ext = os.path.splitext(img['file_name'])
#         extension_counts[ext.lower()] += 1

#     # Print results and statistics
#     print("\n" + "=" * 40)
#     print("VERIFICATION RESULTS:")
#     print(f"Paths fixed in JSON: {fixed_count}")
#     print(f"Files NOT FOUND: {not_found_count}")
#     print("-" * 20)
#     print("Unique extensions and their counts:")
#     for ext, count in extension_counts.items():
#         print(f"  {ext if ext else '[no extension]'}: {count}")
#     print("=" * 40)

#     # Save the cleaned JSON
#     with open(config['output_coco_path'], 'w', encoding='utf-8') as f:
#         json.dump(coco_dict, f, indent=4)

#     print(f"\nClean JSON successfully saved to: {config['output_coco_path']}")


# if __name__ == "__main__":
#     fix_image_paths(CONFIG)

## Step 1 – Build lookup tables from COCO JSON

In [ ]:
# category_id → supercategory label  (only "General body shape" categories)
id_to_label: dict[int, str] = {}
for cat in COCO_DS_DICT['categories']:
    if cat['name'] == 'General body shape':
        id_to_label[cat['id']] = cat['supercategory']
        
log.info("'General body shape' category IDs found: %d", len(id_to_label))

In [ ]:
# image_id → {file_name, width, height}  (filtered: no xray / fake / empty / test)
images_dict: dict[int, dict] = {}
skipped_images = defaultdict(int)   # reason → count

for img in tqdm(COCO_DS_DICT['images'], desc="Parsing COCO images"):
    if 'fishial_extra' not in img:
        skipped_images['no_fishial_extra'] += 1
        continue
    fe = img['fishial_extra']
    skip_reason = None
    if fe.get('xray'):           skip_reason = 'xray'
    elif fe.get('not_a_real_fish'): skip_reason = 'not_a_real_fish'
    elif fe.get('no_fish'):      skip_reason = 'no_fish'
    elif fe.get('test_image'):   skip_reason = 'test_image'
    if skip_reason:
        skipped_images[skip_reason] += 1
        continue
    images_dict[img['id']] = {
        'file_name': img['file_name'],
        'width':     img['width'],
        'height':    img['height'],
    }

log.info("Valid COCO images: %d", len(images_dict))
log.info("Skipped COCO images: %s", dict(skipped_images))

In [ ]:
# drawn_fish_id → annotation info from COCO
new_dict: dict[str, dict] = {}
skipped_anns = defaultdict(int)

for ann in tqdm(COCO_DS_DICT['annotations'], desc="Parsing COCO annotations"):
    if 'category_id' not in ann:
        skipped_anns['no_category_id'] += 1
        continue
    if ann['category_id'] not in id_to_label:
        skipped_anns['wrong_category'] += 1
        continue
    if 'fishial_extra' not in ann or 'drawn_fish_id' not in ann['fishial_extra']:
        skipped_anns['no_drawn_fish_id'] += 1
        continue
    if not ann.get('segmentation'):
        skipped_anns['no_segmentation'] += 1
        continue

    drawn_fish_id = ann['fishial_extra']['drawn_fish_id']
    new_dict[drawn_fish_id] = {
        'image_id':     ann['image_id'],
        'ann_id':       ann['id'],
        'label':        id_to_label[ann['category_id']],
        'segmentation': ann['segmentation'],
    }

log.info("Valid COCO annotations (drawn_fish_id): %d", len(new_dict))
log.info("Skipped COCO annotations: %s", dict(skipped_anns))

## Step 2 – Build lookup tables from the existing FiftyOne dataset

In [ ]:
# drawn_fish_id → {image_id, ann_id, label, fo_id}
existing_fish: dict[str, dict] = {}

# image_id → fo sample id  (for fast lookup when adding polylines to existing images)
image_id_to_fo_id: dict[int, str] = {}

for sample in tqdm(fo_ds, desc="Indexing FiftyOne dataset"):
    image_id = sample['image_id']
    if image_id is not None:
        image_id_to_fo_id[image_id] = sample.id

    field = sample[FIELD_NAME]
    if field is None:
        continue
    for poly in field.polylines:
        drawn_fish_id = poly['drawn_fish_id']
        if drawn_fish_id is None:
            continue
        existing_fish[drawn_fish_id] = {
            'image_id': image_id,
            'ann_id':   poly['ann_id'],
            'label':    poly.label,
            'fo_id':    sample.id,
        }

log.info("Existing drawn_fish_ids in FiftyOne: %d", len(existing_fish))
log.info("Existing image_ids in FiftyOne: %d",    len(image_id_to_fo_id))

## Step 3 – Compute before-merge statistics baseline

In [ ]:
# label → count in FiftyOne BEFORE merge
label_counts_before: dict[str, int] = defaultdict(int)
for info in existing_fish.values():
    label_counts_before[info['label']] += 1

log.info("Label distribution BEFORE merge (top-20):")
for label, cnt in sorted(label_counts_before.items(), key=lambda x: -x[1])[:20]:
    log.info("  %-40s %d", label, cnt)

## Step 4 – Merge

In [ ]:
import mimetypes
mimetypes.add_type("image/webp", ".webp")
mimetypes.add_type("image/jpeg", ".jfif")

# ── Statistics counters ───────────────────────────────────────────────────────
stats = {
    # label updates
    'labels_changed':           0,
    'label_changes_detail':     [],   # list of {drawn_fish_id, old_label, new_label, fo_id}

    # new polylines added to existing FO samples
    'polylines_added_to_existing_sample': 0,

    # brand-new FO samples created
    'new_fo_samples_created':   0,
    'new_fo_sample_ids':        [],

    # skips
    'skipped_image_not_in_valid_list': 0,
    'skipped_missing_file':     0,

    # per-label delta  label → {added, changed}
    'label_delta':              defaultdict(lambda: {'added': 0, 'changed': 0}),
}

# ─────────────────────────────────────────────────────────────────────────────

def _make_polyline(label: str, segmentation: list, width: int, height: int,
                   ann_id: int, drawn_fish_id: str) -> fo.Polyline:
    """Convert COCO segmentation polygon to a normalised FiftyOne Polyline."""
    poly_abs = segmentation[0]          # first (and usually only) polygon
    points   = np.array(poly_abs).reshape(-1, 2).tolist()
    points_norm = [[x / width, y / height] for x, y in points]

    polyline = fo.Polyline(
        label=label,
        points=[points_norm],
        closed=True,
        filled=False,
    )
    polyline['tags']           = [EXPORT_TAG]
    polyline['ann_id']         = ann_id
    polyline['drawn_fish_id']  = drawn_fish_id   # BUG FIX: was tmp_sample['drawn_fish_id']
    return polyline


for new_drawn_fish_id, tmp in tqdm(new_dict.items(), desc="Merging annotations"):

    # ── Case A: drawn_fish_id already exists in FiftyOne ─────────────────────
    if new_drawn_fish_id in existing_fish:
        old_info  = existing_fish[new_drawn_fish_id]
        old_label = old_info['label']
        new_label = tmp['label']

        if old_label != new_label:
            sample = fo_ds[old_info['fo_id']]
            changed = False
            for poly in sample[FIELD_NAME].polylines:
                if poly['drawn_fish_id'] == new_drawn_fish_id:
                    poly['label']                = new_label
                    poly['label_has_changed']    = True
                    poly['label_has_changed_from'] = old_label
                    changed = True
            if changed:
                sample.save()
                stats['labels_changed'] += 1
                stats['label_changes_detail'].append({
                    'drawn_fish_id': new_drawn_fish_id,
                    'old_label':     old_label,
                    'new_label':     new_label,
                    'fo_id':         old_info['fo_id'],
                })
                stats['label_delta'][new_label]['changed'] += 1

    # ── Case B: brand-new drawn_fish_id ──────────────────────────────────────
    else:
        image_id = tmp['image_id']

        # image must pass quality filter
        if image_id not in images_dict:
            stats['skipped_image_not_in_valid_list'] += 1
            continue

        img_meta  = images_dict[image_id]
        file_path = os.path.join(IMAGES_DIR, img_meta['file_name'])

        if not os.path.isfile(file_path):
            log.warning("File not found, skipping: %s", file_path)
            stats['skipped_missing_file'] += 1
            continue

        # Read actual image dimensions (COCO metadata may be stale)
        with Image.open(file_path) as pil_img:
            width, height = pil_img.size

        polyline = _make_polyline(
            label         = tmp['label'],
            segmentation  = tmp['segmentation'],
            width         = width,
            height        = height,
            ann_id        = tmp['ann_id'],
            drawn_fish_id = new_drawn_fish_id,   # BUG FIX
        )

        stats['label_delta'][tmp['label']]['added'] += 1

        # ── Case B1: image already has a FiftyOne sample ──────────────────
        if image_id in image_id_to_fo_id:
            sample = fo_ds[image_id_to_fo_id[image_id]]
            sample[FIELD_NAME].polylines.append(polyline)
            sample.save()
            stats['polylines_added_to_existing_sample'] += 1

        # ── Case B2: create a brand-new FiftyOne sample ───────────────────
        else:
            # BUG FIX 1: use FIELD_NAME, not "polylines"
            # BUG FIX 2: add_sample() BEFORE reading sample.id
            new_sample = fo.Sample(
                filepath = file_path,
                tags     = [EXPORT_TAG],
            )
            
            new_sample[FIELD_NAME] = fo.Polylines(polylines=[polyline])
            new_sample['image_id'] = image_id
            new_sample['width']    = width
            new_sample['height']   = height

            fo_ds.add_sample(new_sample)   # BUG FIX: was sample.save() without add_sample

            # update lookup so the next iteration reuses this sample
            image_id_to_fo_id[image_id] = new_sample.id   # BUG FIX: id is valid only after add_sample
            
            stats['new_fo_samples_created'] += 1
            stats['new_fo_sample_ids'].append(new_sample.id)

log.info("Merge complete.")

## Step 5 – Statistics report

In [ ]:
total_new_polylines = (
    stats['polylines_added_to_existing_sample'] +
    stats['new_fo_samples_created']
)

print("=" * 60)
print("MERGE STATISTICS")
print("=" * 60)
print(f"  Total COCO annotations processed       : {len(new_dict):>8}")
print()
print("  ── Label updates ──────────────────────────────────────")
print(f"  Labels changed (relabelled)             : {stats['labels_changed']:>8}")
print()
print("  ── New polylines added ────────────────────────────────")
print(f"  Polylines added to EXISTING samples     : {stats['polylines_added_to_existing_sample']:>8}")
print(f"  Polylines in brand-new samples          : {stats['new_fo_samples_created']:>8}")
print(f"  Total NEW polylines                     : {total_new_polylines:>8}")
print()
print("  ── New FiftyOne samples ───────────────────────────────")
print(f"  Brand-new FO samples created            : {stats['new_fo_samples_created']:>8}")
print()
print("  ── Skipped ────────────────────────────────────────────")
print(f"  Skipped (image not in valid list)       : {stats['skipped_image_not_in_valid_list']:>8}")
print(f"  Skipped (file missing on disk)          : {stats['skipped_missing_file']:>8}")
print()
print("  ── COCO-level skips ───────────────────────────────────")
for reason, cnt in skipped_anns.items():
    print(f"  Annotations skipped ({reason:<20}): {cnt:>8}")
for reason, cnt in skipped_images.items():
    print(f"  Images skipped     ({reason:<21}): {cnt:>8}")
print("=" * 60)

In [ ]:
# ── Per-label delta table ─────────────────────────────────────────────────────
# Combine before / after counts
label_counts_after: dict[str, int] = defaultdict(int)
for sample in fo_ds:
    field = sample[FIELD_NAME]
    if field is None:
        continue
    for poly in field.polylines:
        label_counts_after[poly.label] += 1

all_labels = sorted(
    set(label_counts_before) | set(label_counts_after),
    key=lambda l: -label_counts_after.get(l, 0)
)

print(f"\n{'Label':<40} {'Before':>8} {'After':>8} {'Δ Added':>8} {'Δ Changed':>10}")
print("-" * 76)
for label in all_labels:
    before  = label_counts_before.get(label, 0)
    after   = label_counts_after.get(label, 0)
    added   = stats['label_delta'][label]['added']
    changed = stats['label_delta'][label]['changed']
    marker  = " ←" if (added or changed) else ""
    print(f"{label:<40} {before:>8} {after:>8} {added:>8} {changed:>10}{marker}")
print("-" * 76)
print(f"{'TOTAL':<40} {sum(label_counts_before.values()):>8} {sum(label_counts_after.values()):>8}")

In [ ]:
# ── Detail log of every label change ─────────────────────────────────────────
if stats['label_changes_detail']:
    print(f"\nDetailed label changes ({len(stats['label_changes_detail'])} total):")
    print(f"  {'drawn_fish_id':<36} {'Old label':<35} {'New label':<35}")
    print("-" * 110)
    for ch in stats['label_changes_detail']:
        print(f"  {ch['drawn_fish_id']:<36} {ch['old_label']:<35} {ch['new_label']:<35}")
else:
    print("No label changes recorded.")

In [ ]:
# ── Save full statistics to JSON ──────────────────────────────────────────────
report = {
    'timestamp':                        datetime.now().isoformat(),
    'coco_path':                        COCO_DS_PATH,
    'fo_dataset':                       FO_DS_NAME,
    'export_tag':                       EXPORT_TAG,
    'total_coco_annotations_processed': len(new_dict),
    'labels_changed':                   stats['labels_changed'],
    'polylines_added_to_existing_samples': stats['polylines_added_to_existing_sample'],
    'new_fo_samples_created':           stats['new_fo_samples_created'],
    'total_new_polylines':              total_new_polylines,
    'skipped_image_not_in_valid_list':  stats['skipped_image_not_in_valid_list'],
    'skipped_missing_file':             stats['skipped_missing_file'],
    'skipped_annotations':              dict(skipped_anns),
    'skipped_images':                   dict(skipped_images),
    'label_changes_detail':             stats['label_changes_detail'],
    'label_delta':                      {k: dict(v) for k, v in stats['label_delta'].items()},
    'label_counts_before':              dict(label_counts_before),
    'label_counts_after':               dict(label_counts_after),
    'new_fo_sample_ids':                stats['new_fo_sample_ids'],
}

report_path = f"merge_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

log.info("Statistics saved to %s", report_path)
print(f"\nFull report saved → {report_path}")

In [ ]:
stats_dict = dict()

for sample in tqdm(fo_ds):

    for polyline in sample['General body shape']['polylines']:
        if polyline['label'] not in stats_dict:
            stats_dict[polyline['label']] = {
                'count': 1,
                'val': 0,
                'train': 0,
                'not_sorted': 0,
            }

        tags = [z.split('_')[0] for z in polyline.tags]
        if 'val' in tags:
            stats_dict[polyline['label']]['val'] += 1
        elif 'train' in tags:
            stats_dict[polyline['label']]['train'] += 1
        else:
            stats_dict[polyline['label']]['not_sorted'] += 1
    # break